# Event Aware Proactive Driver Positioning & Rule-Based Dynamic Pricing
---

## Introduction & Problem Statement

Colombo's ride-hailing business (PickMe, Uber) experiences severe supply-demand imbalances during scheduled, calendar-driven events:

- **Cricket matches** at R. Premadasa Stadium (Z10, 35 000 cap) and SSC Ground (Z4)
- **Poya (full-moon) days** driving temple pilgrimage surges at Kelaniya (Z8) and Gangaramaya (Z9)
- **Southwest Monsoon** (May–September) causing city-wide demand uplift
- **School run windows** (07:00–08:30 and 14:30–16:00 weekdays) in suburban zones

**Three coordinated AI components address this:**

| # | Component | AI Category | Role |
|---|-----------|-------------|------|
| 1 | Demand Knowledge Base | Category 2 – Knowledge Rep. | 10 IF-THEN rules → zone demand scores |
| 2 | Driver Positioning Engine | Category 1 – Search & Opt. | Greedy + SA → driver assignments |
| 3 | Dynamic Pricing Engine | Category 3 – Hybrid | 6 pricing rules + ML demand comparison |


## Setup & Data Loading

In [ ]:
import sys, json, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
warnings.filterwarnings('ignore')

ROOT = Path().resolve()
sys.path.insert(0, str(ROOT / 'src'))

travel_matrix = pd.read_csv(ROOT / 'data/processed/travel_time_matrix.csv', index_col=0)
print('Travel matrix shape:', travel_matrix.shape)
travel_matrix.round(1)


In [ ]:
weather_df = pd.read_csv(ROOT / 'data/processed/historical_weather_clean.csv')
poya_df    = pd.read_csv(ROOT / 'data/processed/poya_calendar.csv')
cricket_df = pd.read_csv(ROOT / 'data/processed/cricket_fixtures.csv')
demand_df  = pd.read_csv(ROOT / 'data/processed/demand_dataset.csv')
print(f'Weather records    : {len(weather_df):,}')
print(f'Poya holidays      : {len(poya_df)}')
print(f'Cricket fixtures   : {len(cricket_df)}')
print(f'Demand dataset rows: {len(demand_df):,}')
print(f'Demand splits      : {demand_df.split.value_counts().to_dict()}')


## Event Scraper Demo

The event_scraper module scrapes upcoming events from LankaEvents.lk and Eventbrite. Venue names are geocoded to the ten demand zones using keyword matching (target precision >= 80%).


In [ ]:
from event_scraper import get_upcoming_events, geocode_venue_to_zone

events = get_upcoming_events(use_fallback_only=True)
ev_df  = pd.DataFrame(events)
print(f'Loaded {len(ev_df)} events (fallback mode)')
ev_df[['name','venue','zone','event_type','crowd_estimate']].head(8)


In [ ]:
# Geocoder spot-checks
spot = [
    ('R. Premadasa Stadium', 'Z10'),
    ('SSC Ground, Maitland Crescent', 'Z4'),
    ('Gangaramaya Temple', 'Z9'),
    ('BMICH, Bauddhaloka Mawatha', 'Z3'),
    ('Kelaniya Raja Maha Viharaya', 'Z8'),
    ('SLECC, Colombo 02', 'Z2'),
]
n_pass = 0
for venue, expected in spot:
    pred = geocode_venue_to_zone(venue)
    ok   = pred == expected
    if ok: n_pass += 1
    print(f'  [{"PASS" if ok else "FAIL"}] {venue:<40} -> {pred} (expected {expected})')
print(f'Precision: {n_pass}/{len(spot)} = {n_pass/len(spot):.0%}')


## Demand Knowledge Base (Category 2)

10 IF-THEN production rules encode expert knowledge of Colombo demand patterns.
Rules are additive, clamped to [−50%, +150%].

| Rule | Condition | Uplift |
|------|-----------|--------|
| R01 | Cricket at Premadasa, ≤20 min to end | Z10 +80%, Z3/Z4 +40% |
| R02 | Cricket at SSC, ≤20 min to end | Z4 +80%, Z3 +40% |
| R03 | Heavy rain >5mm/hr | All +40% |
| R04 | Moderate rain 2.5–5mm/hr | All +20% |
| R05 | Poya day | Z8 +90%, Z1 +30% |
| R06 | Poya day | Z9 +70%, Z2 +30% |
| R07 | Morning school run (07:00–08:30 weekday) | Z6 +50%, Z7 +45% |
| R08 | Afternoon school run (14:30–16:00 weekday) | Z6 +50%, Z7 +45% |
| R09 | Concert at BMICH, crowd >1000 | Z3 +70%, Z9 +30% |
| R10 | Early morning 01:00–05:00 | All −30% |


In [ ]:
from demand_rule_base import evaluate_demand, ZONES

ctx_cricket = {
    'event_type': 'cricket', 'venue_zone': 'Z10',
    'minutes_to_event_end': 15, 'crowd_estimate': 35000,
    'current_rainfall_mm': 0.0, 'is_poya_day': False,
    'hour': 21, 'minute': 30, 'day_of_week': 5,
}
r_cricket = evaluate_demand(ctx_cricket)
print('Cricket scenario — fired rules:', r_cricket['fired_rules'])
for z, u in r_cricket['demand_uplifts'].items():
    bar  = '#' * int(abs(u) * 20)
    sign = '+' if u >= 0 else ''
    print(f'  {z}: {sign}{u:+.2f}  {bar}')


In [ ]:
ctx_poya_rain = {
    'is_poya_day': True, 'current_rainfall_mm': 5.5,
    'event_type': None, 'venue_zone': None,
    'hour': 16, 'minute': 0, 'day_of_week': 0,
    'minutes_to_event_end': 999, 'crowd_estimate': 0,
}
r_poya = evaluate_demand(ctx_poya_rain)
print('Poya + heavy rain — fired rules:', r_poya['fired_rules'])
for z, u in r_poya['demand_uplifts'].items():
    bar = '#' * int(abs(u) * 20)
    print(f'  {z}: {u:+.2f}  {bar}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, result, title in zip(
    axes,
    [r_cricket, r_poya],
    ['Cricket: Post-match Premadasa (R01)', 'Vesak Poya + Heavy Rain (R03, R05, R06)']
):
    uplifts = [result['demand_uplifts'][z] for z in ZONES]
    colors  = ['#E63946' if u > 0 else '#457B9D' for u in uplifts]
    ax.bar(ZONES, uplifts, color=colors, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel('Demand Uplift'); ax.set_ylim(-0.6, 1.6)
    ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'outputs/figures/demand_uplift_comparison.png', bbox_inches='tight')
plt.show()


### Rule Base vs Learned Demand Model

The rule base is compared to a Gradient Boosted Trees (GBT) model trained on the file demand_dataset.csv. The GBT captures nonlinear rain-peak interactions that the additive rule base cannot express (super-additivity +0.264 in held-out test).


In [ ]:
comp_model = pd.read_csv(ROOT / 'outputs/reports/demand_model_comparison.csv')
print('Rule base vs Learned — test set performance:')
print(comp_model[['Model','MAE','RMSE','R2','CV RMSE']].to_string(index=False))


In [ ]:
from IPython.display import Image
Image(str(ROOT / 'outputs/figures/demand_model_feature_importance.png'), width=700)


## Driver Positioning Engine (Category 1)

**Greedy Best-First (primary):** Iterates deficit zones by urgency (demand × deficit descending), assigns the closest eligible idle driver. Hard constraint: no driver moved >10 minutes.

**Simulated Annealing (fallback):** Warm-starts from greedy, uses Kirkpatrick cooling τ_{k+1} = 0.95 × τ_k.


In [ ]:
from positioning_engine import run_positioning_engine

drivers  = pd.read_csv(ROOT / 'data/processed/driver_fleet_poya_seed1.csv')
poya_ctx = {'is_poya_day': True, 'current_rainfall_mm': 0.0,
            'event_type': None, 'venue_zone': None,
            'hour': 10, 'minute': 0, 'day_of_week': 3,
            'minutes_to_event_end': 999, 'crowd_estimate': 0}
demand_r = evaluate_demand(poya_ctx)

rg = run_positioning_engine(drivers, demand_r['demand_uplifts'], travel_matrix, use_sa=False)
rs = run_positioning_engine(drivers, demand_r['demand_uplifts'], travel_matrix, use_sa=True)

print(f'GREEDY — {len(rg["assignments"])} assignments | violations={rg["constraint_violations"]} | {rg["computation_time_sec"]:.3f}s')
print(f'SA     — {len(rs["assignments"])} assignments | violations={rs["constraint_violations"]} | {rs["computation_time_sec"]:.3f}s')
print()
for a in rg['assignments']:
    print(f'  {a["driver_id"]}  {a["from_zone"]} -> {a["to_zone"]}  ({a["travel_time_min"]:.1f} min)')


In [ ]:
# Coverage: before vs after
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, cov, title in zip(
    axes, [rg['pre_coverage'], rg['post_coverage']],
    ['Before Repositioning', 'After Greedy Repositioning']
):
    vals   = [cov.get(z, 0.0) for z in ZONES]
    colors = ['#2A9D8F' if v >= 1.0 else '#E76F51' for v in vals]
    ax.bar(ZONES, vals, color=colors, edgecolor='white')
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Full coverage')
    ax.set_title(title, fontweight='bold'); ax.set_ylim(0, 1.4)
    ax.set_ylabel('Coverage Ratio'); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'outputs/figures/coverage_before_after.png', bbox_inches='tight')
plt.show()


## 6. Dynamic Pricing Engine (Category 3 – Hybrid)

| Rule | Condition | Multiplier |
|------|-----------|------------|
| P04 | Large event (>5000 crowd) ending ≤20 min | 1.30× |
| P03 | High demand + gap >60% | 1.50× |
| P02 | High demand + gap 31–60% | 1.30× |
| P01 | High demand + gap 1–30% | 1.15× |
| P06 | High demand + gap = 0 (repositioning resolved) | 1.00× |
| P05 | Excess supply, low demand | 0.85× |

**Caps:** Hard cap 2.0× | Fair-use cap 1.8× (Poya + heavy rain combined)


In [ ]:
from pricing_engine import compute_price_multipliers

pricing = compute_price_multipliers(demand_r, rg['coverage_gap'], poya_ctx)
print('Per-zone multipliers (Poya scenario, post-greedy):')
for z, m in pricing['multipliers'].items():
    rule = pricing['applied_rules'][z] or 'none'
    bar  = '*' * int((m - 0.8) * 40)
    print(f'  {z}: {m:.2f}x  rule={rule:<4}  {bar}')


In [ ]:
mults  = [pricing['multipliers'][z] for z in ZONES]
colors = ['#E63946' if m > 1.0 else ('#457B9D' if m < 1.0 else '#aaa') for m in mults]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(ZONES, mults, color=colors, edgecolor='white')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
ax.set_title('Dynamic Pricing Multipliers — Poya Day Scenario', fontweight='bold')
ax.set_ylabel('Fare Multiplier'); ax.set_ylim(0.7, 1.7)
for i, m in enumerate(mults):
    ax.text(i, m + 0.02, f'{m:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'outputs/figures/pricing_multipliers_poya.png', bbox_inches='tight')
plt.show()


## 7. Simulation & Evaluation (30 Scenarios × 3 Variants)

Results loaded from pre-computed `evaluation_results.csv`. The pipeline was run with `py -m src.evaluation` (90 total runs: 30 scenarios × baseline / greedy / SA).


In [ ]:
results_df = pd.read_csv(ROOT / 'data/processed/evaluation_results.csv')
comp_table = pd.read_csv(ROOT / 'data/processed/comparison_table.csv')
print(f'Rows: {len(results_df)}')
print(comp_table.to_string(index=False))


In [ ]:
from config import METRICS_TARGETS as T
greedy_df = results_df[results_df.variant == 'greedy']

criteria = [
    ('Wait Time Reduction',      'wait_time_reduction_pct',       '>=', T['wait_time_reduction_pct']),
    ('Surge Prevention Rate',    'surge_prevention_rate',         '>=', T['surge_prevention_rate']),
    ('Avg Surge Multiplier',     'avg_surge_multiplier',          '<=', T['avg_surge_multiplier_max']),
    ('Gini Coefficient',         'gini_coefficient',              '<',  T['gini_target']),
    ('Constraint Satisfaction',  'constraint_satisfaction_rate',  '>=', T['constraint_satisfaction']),
    ('Computation Time (s)',     'computation_time_sec',          '<',  T['computation_time_max_sec']),
]

print(f'{"Metric":<30} {"Achieved":>10} {"Target":>10} {"Status":>8}')
print('-' * 62)
for label, col, op, target in criteria:
    val = greedy_df[col].mean()
    if op == '>=':   ok = val >= target
    elif op == '<=': ok = val <= target
    else:            ok = val < target
    print(f'{label:<30} {val:>10.4f} {target:>10.4f} {"PASS" if ok else "FAIL":>8}')


In [ ]:
# Comparison bar charts
scenarios = ['monsoon', 'cricket', 'poya']
variants  = ['baseline', 'greedy', 'sa']
vc        = {'baseline': '#aaa', 'greedy': '#2A9D8F', 'sa': '#E63946'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_p = [
    ('surge_prevention_rate', 'Surge Prevention Rate', (0, 1.1)),
    ('gini_coefficient',      'Gini Coefficient',     (0, 0.5)),
    ('wait_time_reduction_pct','Wait Time Reduction', (-0.1, 0.6)),
]
x = np.arange(3); w = 0.25
for ax, (col, label, ylim) in zip(axes, metrics_p):
    for i, v in enumerate(variants):
        vals = [results_df[(results_df.variant==v)&(results_df.scenario_type==s)][col].mean()
                for s in scenarios]
        ax.bar(x + i*w, vals, w, label=v.capitalize(), color=vc[v], edgecolor='white')
    ax.set_xticks(x+w); ax.set_xticklabels(['Monsoon','Cricket','Poya'])
    ax.set_title(label, fontweight='bold'); ax.set_ylim(*ylim); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ROOT / 'outputs/figures/evaluation_comparison.png', bbox_inches='tight')
plt.show()


## 8. Qualitative Case Studies

### Case Study 1: R. Premadasa Stadium — T20I Post-Match
*Source: Watchdog Sri Lanka — 30+ minute customer wait times documented post-Colombo cricket matches*

Conditions: T20I Saturday night, 35 000 crowd, 15 minutes to match end, no rain.


In [ ]:
from simulation import run_scenario, run_scenario_baseline

with open(ROOT / 'data/scenarios/cricket/cricket_seed0.json') as f:
    cs1_cfg = json.load(f)

cs1_base   = run_scenario_baseline(cs1_cfg, travel_matrix)
cs1_greedy = run_scenario(cs1_cfg, travel_matrix)

print('Case Study 1: Premadasa Post-match T20I')
print(f'  Fired rules     : {cs1_greedy["demand"]["fired_rules"]}')
print(f'  Assignments     : {len(cs1_greedy["positioning"]["assignments"])}')
print(f'  Surge prev (B)  : {cs1_base["metrics"]["surge_prevention_rate"]:.0%}')
print(f'  Surge prev (G)  : {cs1_greedy["metrics"]["surge_prevention_rate"]:.0%}')
print(f'  Z10 multiplier  : {cs1_greedy["pricing"]["multipliers"]["Z10"]}x')
print(f'  Constraint viol.: {cs1_greedy["positioning"]["constraint_violations"]}')
print(f'  Computation (s) : {cs1_greedy["metrics"]["computation_time_sec"]:.3f}')


### Case Study 2: Monsoon Friday — Kollupitiya (June 2024)
*Source: Watchdog Sri Lanka survey — documented 30-minute wait times during SW Monsoon evenings*

Conditions: Friday afternoon, school-run overlap with heavy monsoon rain (7.5 mm/hr).


In [ ]:
with open(ROOT / 'data/scenarios/monsoon/monsoon_seed0.json') as f:
    cs2_cfg = json.load(f)

cs2_base   = run_scenario_baseline(cs2_cfg, travel_matrix)
cs2_greedy = run_scenario(cs2_cfg, travel_matrix)

print('Case Study 2: Monsoon Friday Afternoon')
print(f'  Fired rules     : {cs2_greedy["demand"]["fired_rules"]}')
print(f'  Baseline wait   : {cs2_base["metrics"]["baseline_wait_time_min"]:.2f} min')
print(f'  System wait     : {cs2_greedy["metrics"]["system_wait_time_min"]:.2f} min')
print(f'  Wait reduction  : {cs2_greedy["metrics"]["wait_time_reduction_pct"]:.1%}')
print(f'  Surge prev.     : {cs2_greedy["metrics"]["surge_prevention_rate"]:.0%}')
print(f'  Z2 multiplier   : {cs2_greedy["pricing"]["multipliers"]["Z2"]}x  (Kollupitiya)')


### Case Study 3: Vesak Poya Day + Heavy Rain (Fair-Use Cap Test)
*Source: News coverage of Vesak-day surge pricing complaints in Colombo (2025)*

Conditions: Poya day + heavy rain (5.5 mm/hr) — triggers combined 1.80x fair-use cap.


In [ ]:
with open(ROOT / 'data/scenarios/poya/poya_seed6.json') as f:
    cs3_cfg = json.load(f)
print('Scenario:', cs3_cfg['description'])

cs3_r = run_scenario(cs3_cfg, travel_matrix)
mults = cs3_r['pricing']['multipliers']

print(f'  Fired rules       : {cs3_r["demand"]["fired_rules"]}')
print(f'  Z8 (Kelaniya)     : {mults["Z8"]}x  (fair cap 1.80x: {mults["Z8"] <= 1.80})')
print(f'  Z9 (Gangaramaya)  : {mults["Z9"]}x')
print(f'  Max multiplier    : {max(mults.values())}x  (hard cap = 2.00x)')
print(f'  Fair-use cap held : {max(mults.values()) <= 1.80}')


## 9. Critical Reflection & Limitations

### Strengths
- **Explainability:** Every pricing and positioning decision traces to a specific rule.
- **Constraint satisfaction:** 100% of repositioning assignments respect the 10-min limit.
- **Surge prevention:** 95% mean across all 30 scenarios (target >= 60%).
- **Speed:** Mean 0.43s per cycle (target < 5s) — real-time viable.

### Limitations & Trade-offs

| Issue | Impact | Mitigation |
|-------|--------|------------|
| Synthetic demand data | Validates logic, not real patterns | Use PickMe/Uber trip logs |
| Wait time 16.9% < 25% target | M6 fails | Queuing-theory wait model |
| Gini 0.227 > 0.20 target | M3 fails | Multi-cycle look-ahead |
| Additive rule base | Cannot capture nonlinear interactions | GBT model (R²=0.899) |

### Algorithm Comparison
| Aspect | Greedy | SA |
|--------|--------|----|
| Speed | ~0.4s | ~1.5s |
| Optimality | Local | Better global |
| Determinism | Yes | Stochastic |
| Constraint violations | 0 | 0 |

### Ethical Considerations
Dynamic pricing risks amplifying costs for passengers in low-income zones during crises. The fair-use cap (1.80x) and discount rule (0.85x) partially mitigate this. A socially-aware extension would cap multipliers by neighbourhood income index.

### Future Work
- Real ride data for ground-truth validation
- Online rule refinement from historical ride patterns
- 30-minute ahead horizon for repositioning
- Deep RL comparison for the positioning component


In [ ]:
print('Notebook complete.')
print('All outputs saved to outputs/figures/ and outputs/reports/')
